# One-Sample FEA Inspection Notebook

This notebook exercises the public `src.interfaces` surface only.
It uses a temporary SQLite database and temporary artifact directories, so nothing is written into the source tree.

## Setup

In [ ]:
    import json
    import logging
    import sqlite3
    import sys
    import tempfile
    from pathlib import Path

    def _discover_module_root() -> Path:
        """Find the module root regardless of the notebook launch directory."""

        cwd = Path.cwd().resolve()
        candidates = [cwd, cwd.parent, cwd / "code_base" / "fea_cad_one_sample"]
        for candidate in candidates:
            if (candidate / "src").is_dir() and (candidate / "README.md").is_file():
                return candidate
        raise RuntimeError(f"Could not find fea_cad_one_sample module root from {cwd}")

    MODULE_ROOT = _discover_module_root()
    if str(MODULE_ROOT) not in sys.path:
        sys.path.insert(0, str(MODULE_ROOT))

    logging.basicConfig(level=logging.DEBUG, format="%(name)s | %(levelname)s | %(message)s")

    from src import interfaces as api

    workspace_root = Path(tempfile.mkdtemp(prefix="fea_cad_one_sample_inspection_"))
    db_path = workspace_root / "inspection.db"
    connection_string = f"sqlite:///{db_path}"
    sample_id = "sample-inspection-001"
    output_root = workspace_root / "artifacts"
    output_root.mkdir(parents=True, exist_ok=True)
    pipeline_config = api.PipelineConfig(
        config_name="config_gpt_5_4_mini.yaml",
        config_path=MODULE_ROOT / "src" / "copied_from_cadcodeverify" / "configs" / "config_gpt_5_4_mini.yaml",
        output_root=output_root,
        force=True,
    )

    print("
" + "─" * 60)
    print("[STAGE] setup")
    print(f"  → module root : {MODULE_ROOT}")
    print(f"  → workspace   : {workspace_root}")
    print(f"  → db path     : {db_path}")
    print(f"  → config name : {pipeline_config.config_name}")
    print("  ✓ logging     : DEBUG")
    print("─" * 60 + "
")

## Test Input

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] test_input")
    print(f"  → sample_id   : {sample_id}")
    print(f"  → connection  : {connection_string}")
    print(f"  → output_root : {output_root}")

    with sqlite3.connect(db_path) as conn:
        conn.execute(
            "CREATE TABLE master_metadata (id TEXT PRIMARY KEY, expert_prompt TEXT, non_expert_prompt TEXT)"
        )
        conn.execute(
            "CREATE TABLE ground_truth_code (id TEXT PRIMARY KEY, python_code TEXT)"
        )
        conn.execute(
            "INSERT INTO master_metadata (id, expert_prompt, non_expert_prompt) VALUES (?, ?, ?)",
            (
                sample_id,
                "Design a cantilever bracket with a fixed support and a 200 N downward load.",
                "Design a simple support part.",
            ),
        )
        conn.execute(
            "INSERT INTO ground_truth_code (id, python_code) VALUES (?, ?)",
            (
                sample_id,
                "import cadquery as cq
result = cq.Workplane('XY').box(10, 5, 2)",
            ),
        )
        conn.commit()

    print("  ✓ seeded     : master_metadata + ground_truth_code")
    print("  ✓ config     : PipelineConfig ready")
    print("─" * 60 + "
")

## Public Entry Check

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] public_entry_check")

    required_names = [
        "CADSample",
        "PipelineConfig",
        "LoadCase",
        "ManualFEAReport",
        "PipelineSummary",
        "inspect_schema",
        "load_sample",
        "generate_original_code",
        "execute_and_export_cadquery",
        "build_fea_prompt",
        "write_load_case",
        "generate_fea_ready_code",
        "render_views",
        "build_comparison_artifacts",
        "write_freecad_instructions",
        "write_manual_fea_report_template",
        "write_post_fea_prompt",
        "run_full_pipeline",
    ]
    missing = [name for name in required_names if not hasattr(api, name)]
    print(f"  → required  : {len(required_names)} public names")
    print(f"  → missing   : {missing if missing else 'none'}")
    assert not missing, f"Missing public names: {missing}"
    print("  ✓ public API: src.interfaces exports the expected surface")
    print("─" * 60 + "
")

## Stage Inspection

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] schema_and_sample")

    schema_summary = api.inspect_schema(connection_string)
    sample = api.load_sample(connection_string, sample_id=sample_id)

    print(f"  → dialect   : {schema_summary['dialect']}")
    print(f"  → tables    : {[table['name'] for table in schema_summary['tables']]}")
    print(f"  → sample id : {sample.sample_id}")
    print(f"  → variant   : {sample.prompt_variant}")
    print(f"  → metadata  : {sorted(sample.metadata.keys())}")
    print(f"  → prompt    : {sample.prompt}")

    assert schema_summary['table_count'] == 2
    assert sample.sample_id == sample_id
    assert sample.prompt_variant == 'expert'
    assert 'bracket' in sample.prompt.lower()
    print("  ✓ schema    : inspected")
    print("  ✓ sample    : loaded")
    print("─" * 60 + "
")

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] fea_artifacts")

    load_case_path = output_root / '02_fea_ready' / 'load_case.json'
    manual_report_path = output_root / '04_manual_freecad_fea' / 'fea_report.json'
    comparison_dir = output_root / '03_comparison'
    post_fea_dir = output_root / '05_post_fea_refinement'

    load_case = api.write_load_case(sample.sample_id, load_case_path, force=True)
    fea_prompt = api.build_fea_prompt(sample, load_case)
    manual_report = api.write_manual_fea_report_template(sample.sample_id, manual_report_path, force=True)
    comparison_paths = api.build_comparison_artifacts(
        sample.prompt,
        fea_prompt,
        comparison_dir,
        notes={"source": "notebook inspection", "sample_id": sample.sample_id},
        force=True,
    )
    post_fea_paths = api.write_post_fea_prompt(
        sample.sample_id,
        load_case,
        manual_report_path,
        post_fea_dir,
        force=True,
    )

    print(f"  → load case : {load_case_path}")
    print(f"  → prompt ln : {len(fea_prompt.splitlines())}")
    print(f"  → report    : {manual_report_path}")
    print(f"  → compare   : {comparison_paths}")
    print(f"  → post-FEA  : {post_fea_paths}")
    print(f"  → max VM    : {load_case.requirements['max_von_mises_pa']}")

    assert load_case.requirements['max_von_mises_pa'] == 138_000_000
    assert load_case_path.exists()
    assert manual_report_path.exists()
    assert (comparison_dir / 'prompt_diff.md').exists()
    assert (comparison_dir / 'geometry_diff_notes.md').exists()
    assert (post_fea_dir / 'fea_feedback_prompt.txt').exists()
    assert (post_fea_dir / 'comparison_after_fea.md').exists()
    print("  ✓ load case : written")
    print("  ✓ prompt    : built")
    print("  ✓ reports   : written")
    print("─" * 60 + "
")

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] cad_execution_and_rendering")

    cad_output_dir = output_root / '01_original'
    views_dir = cad_output_dir / 'views'
    cad_script = "
".join([
        "import cadquery as cq",
        "result = cq.Workplane('XY').box(20, 10, 5)",
    ])

    cad_result = api.execute_and_export_cadquery(cad_script, cad_output_dir, basename='sample_box', force=True)
    view_paths = api.render_views(Path(cad_result['stl_path']), views_dir, force=True)

    print(f"  → step path : {cad_result['step_path']}")
    print(f"  → stl path  : {cad_result['stl_path']}")
    print(f"  → log path  : {cad_result['execution_log_path']}")
    print(f"  → views     : {view_paths}")

    assert Path(cad_result['step_path']).exists()
    assert Path(cad_result['stl_path']).exists()
    assert Path(cad_result['execution_log_path']).exists()
    assert all(Path(path).exists() for path in view_paths.values())
    print("  ✓ cadquery  : executed and exported")
    print("  ✓ views     : rendered")
    print("─" * 60 + "
")

## Assertions

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] assertions")

    assert isinstance(sample, api.CADSample)
    assert isinstance(load_case, api.LoadCase)
    assert isinstance(manual_report, api.ManualFEAReport)
    assert isinstance(schema_summary, dict)
    assert isinstance(comparison_paths, dict)
    assert isinstance(post_fea_paths, dict)
    assert len(view_paths) == 4

    print(f"  ✓ sample    : {type(sample).__name__}")
    print(f"  ✓ load_case : {type(load_case).__name__}")
    print(f"  ✓ report    : {type(manual_report).__name__}")
    print(f"  ✓ views     : {sorted(view_paths.keys())}")
    print("─" * 60 + "
")

## Failure Case

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] failure_case")

    try:
        api.load_sample(connection_string, sample_id=sample_id, random=True)
    except Exception as exc:
        print(f"  ✗ error type : {type(exc).__name__}")
        print(f"  ✗ message    : {exc}")
        assert isinstance(exc, ValueError)
    else:
        raise AssertionError("Expected load_sample to reject conflicting selection flags.")

    print("  ✓ failure   : validated")
    print("─" * 60 + "
")

## Artifact Inspection

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] artifact_inspection")

    all_files = sorted(path for path in workspace_root.rglob('*') if path.is_file())
    for path in all_files:
        rel_path = path.relative_to(workspace_root)
        print(f"  • {rel_path} ({path.stat().st_size} bytes)")

    print(f"  → file count : {len(all_files)}")
    print(f"  → workspace  : {workspace_root}")
    print("  ✓ artifacts  : inspected")
    print("─" * 60 + "
")

## Debugging Notes

- If schema inspection fails, confirm the temporary SQLite tables match the expected names: `master_metadata` and `ground_truth_code`.
- If CadQuery export fails, check that the script assigns a `result` object.
- If view rendering fails, confirm the STL path exists before calling `render_views()`.
- The setup cell auto-discovers the module root, so the notebook should work whether it is launched from the repo root or the `notebooks/` directory.
- All outputs in this notebook are written under a temporary workspace, not under `code_base/`.

## Summary

In [ ]:
    print("
" + "─" * 60)
    print("[STAGE] summary")

    print(f"  → workspace  : {workspace_root}")
    print(f"  → sample id  : {sample.sample_id}")
    print(f"  → artifacts  : {len(all_files)} files")
    print(f"  → views      : {sorted(view_paths.keys())}")
    print(f"  → comparison : {comparison_dir}")
    print(f"  → post-FEA   : {post_fea_dir}")
    print("  ✓ notebook   : inspection flow complete")
    print("─" * 60 + "
")